# Stage 1: CIFAR-100 data pipeline

This notebook builds deterministic 80/10/10 class-stratified splits, train-only normalization, a P×K batch sampler, fixed validation triplets, and a saved augmentation preview. The reusable implementation lives in `src/data.py`. Run from the project directory; the first run downloads CIFAR-100 into `data/`.

In [ ]:
from pathlib import Path
import json
import logging
import numpy as np
import matplotlib.pyplot as plt

from src.data import get_splits, get_loaders, get_val_triplets, _load_cifar100

logging.basicConfig(level=logging.INFO, format='%(message)s')
SEED = 42
DATA_ROOT = Path('data')
SPLIT_DIR = Path('splits')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Build the pool and deterministic splits

The global index space is the torchvision training set (0–49,999) followed by the original test set (50,000–59,999). The split JSON stores all indices and canonical little-endian int64 SHA-256 hashes.

In [ ]:
train_source, test_source, pool_labels = _load_cifar100(DATA_ROOT, download=True)
class_names = train_source.classes
splits = get_splits(seed=SEED, data_root=DATA_ROOT, split_dir=SPLIT_DIR)
split_metadata = json.loads((SPLIT_DIR / f'seed{SEED}.json').read_text(encoding='utf-8'))
print('Pool size:', len(pool_labels))
print('Split sizes:', split_metadata['counts'])
print('SHA-256:')
for name, digest in split_metadata['sha256'].items():
    print(f'  {name}: {digest}')

## Split and sampler exit checks

These assertions check per-class counts, pairwise disjointness, same-seed reproducibility, different-seed hashes, and exact P=32, K=8 composition. A 48,000-example train set gives 187 full PK batches per epoch (47,872 sampled positions).

In [ ]:
splits_again = get_splits(seed=SEED, data_root=DATA_ROOT, split_dir=SPLIT_DIR)
splits_other_seed = get_splits(seed=SEED + 1, data_root=DATA_ROOT, split_dir=SPLIT_DIR)
hashes_again = json.loads((SPLIT_DIR / f'seed{SEED}.json').read_text(encoding='utf-8'))['sha256']
hashes_other = json.loads((SPLIT_DIR / f'seed{SEED + 1}.json').read_text(encoding='utf-8'))['sha256']

for name, expected_count in {'train': 480, 'val': 60, 'test': 60}.items():
    per_class = np.bincount(pool_labels[splits[name]], minlength=100)
    assert len(splits[name]) == expected_count * 100
    assert np.all(per_class == expected_count), (name, np.unique(per_class, return_counts=True))
assert set(splits['train']).isdisjoint(splits['val'])
assert set(splits['train']).isdisjoint(splits['test'])
assert set(splits['val']).isdisjoint(splits['test'])
assert all(np.array_equal(splits[name], splits_again[name]) for name in ('train', 'val', 'test'))
assert hashes_again == split_metadata['sha256']
assert all(hashes_other[name] != hashes_again[name] for name in ('train', 'val', 'test'))
print('Split sizes, class balance, disjointness, and reproducibility: OK')

loaders = get_loaders(seed=SEED, data_root=DATA_ROOT, split_dir=SPLIT_DIR, P=32, K=8)
local_train_labels = pool_labels[splits['train']]
for pk_indices in loaders['train'].batch_sampler:
    pk_classes, pk_counts = np.unique(local_train_labels[pk_indices], return_counts=True)
    assert len(pk_indices) == 256 and len(pk_classes) == 32 and np.all(pk_counts == 8)
train_images, train_labels = next(iter(loaders['train']))
batch_labels = train_labels.numpy()
classes, counts = np.unique(batch_labels, return_counts=True)
assert len(batch_labels) == 32 * 8
assert len(classes) == 32
assert np.all(counts == 8)
assert len(loaders['train']) == 48000 // (32 * 8) == 187
print(f'All {len(loaders["train"])} PK batches verified: 32 classes × 8 images each')

## Fixed validation triplets

Each row stores global pool indices `(anchor, positive, negative)`. Anchors and positives share a validation label; negatives have a different validation label.

In [ ]:
val_triplets = get_val_triplets(seed=SEED, n_triplets=10_000, data_root=DATA_ROOT, split_dir=SPLIT_DIR)
assert val_triplets.shape == (10_000, 3)
assert np.all(np.isin(val_triplets, splits['val']))
assert np.all(pool_labels[val_triplets[:, 0]] == pool_labels[val_triplets[:, 1]])
assert np.all(pool_labels[val_triplets[:, 0]] != pool_labels[val_triplets[:, 2]])
print('Saved splits/val_triplets_seed42.npy:', val_triplets.shape)

## Save a grid of 64 augmented training images

The preview reverses train normalization for display and labels each image with its CIFAR-100 class name.

In [ ]:
normalizer = loaders['train'].dataset.transform.transforms[-1]
mean = np.asarray(normalizer.mean, dtype=np.float32)
std = np.asarray(normalizer.std, dtype=np.float32)
images = train_images[:64].permute(0, 2, 3, 1).numpy() * std + mean
fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    ax.imshow(np.clip(images[i], 0.0, 1.0))
    ax.set_title(class_names[int(train_labels[i])], fontsize=7)
    ax.axis('off')
fig.tight_layout()
grid_path = OUTPUT_DIR / 'augmented_train_grid.png'
fig.savefig(grid_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved:', grid_path.resolve())